In [2]:
# ── Cell 1: Install ──────────────────────────────────────────────
!pip install -U qwen-tts -q
!pip install fastapi uvicorn pyngrok nest-asyncio soundfile -q
# flash-attn removed — using torch SDPA instead (works natively on T4)

import torch
print(f"✅ CUDA available : {torch.cuda.is_available()}")
print(f"✅ GPU name       : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 97.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-cor

In [3]:
# ── Cell 2: Load model ───────────────────────────────────────────
import torch
from qwen_tts import Qwen3TTSModel
import soundfile as sf

print("Loading Qwen3-TTS VoiceDesign 1.7B...")

model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",   # ← flash_attention_2 replaced with sdpa
)

print("✅ Model loaded!")

# Quick test
test_wavs, test_sr = model.generate_voice_design(
    text="Kaggle backend is ready.",
    language="English",
    instruct="Clear, neutral male voice.",
)
sf.write("/kaggle/working/test.wav", test_wavs[0], test_sr)
print("✅ Test passed — Kaggle backend ready.")

2026-06-02 13:54:04.588891: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780408444.825284      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780408444.892894      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780408445.451261      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780408445.451301      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780408445.451304      58 computation_placer.cc:177] computation placer alr


********
********
 
Loading Qwen3-TTS VoiceDesign 1.7B...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

✅ Model loaded!


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


✅ Test passed — Kaggle backend ready.


In [9]:
# ── Cell 3: FastAPI server (identical to Colab except PLATFORM) ──
import os, uuid, base64, nest_asyncio, io
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import soundfile as sf
import uvicorn

nest_asyncio.apply()

PLATFORM = "kaggle"

app = FastAPI(title=f"Qwen3-TTS API — {PLATFORM.upper()}")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class GenerateRequest(BaseModel):
    text: str
    instruct: str
    language: str = "English"

class GenerateResponse(BaseModel):
    audio_b64: str
    sample_rate: int
    platform: str
    duration_seconds: float

SUPPORTED_LANGUAGES = [
    "English", "Chinese", "Japanese", "Korean",
    "German", "French", "Russian", "Portuguese",
    "Spanish", "Italian"
]

@app.get("/")
def health():
    return {
        "status": "ok",
        "platform": PLATFORM,
        "model": "Qwen3-TTS-12Hz-1.7B-VoiceDesign",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "supported_languages": SUPPORTED_LANGUAGES,
    }

@app.post("/generate", response_model=GenerateResponse)
def generate_voice(req: GenerateRequest):
    if not req.text.strip():
        raise HTTPException(status_code=400, detail="Text cannot be empty.")
    if len(req.text) > 500:
        raise HTTPException(status_code=400, detail="Text too long. Max 500 characters.")
    if not req.instruct.strip():
        raise HTTPException(status_code=400, detail="Instruct cannot be empty.")
    if len(req.instruct) > 300:
        raise HTTPException(status_code=400, detail="Instruct too long. Max 300 characters.")
    if req.language not in SUPPORTED_LANGUAGES:
        raise HTTPException(
            status_code=400,
            detail=f"Unsupported language. Choose from: {', '.join(SUPPORTED_LANGUAGES)}"
        )

    try:
        wavs, sr = model.generate_voice_design(
            text=req.text,
            language=req.language,
            instruct=req.instruct,
        )

        audio_array = wavs[0]
        duration = len(audio_array) / sr

        buf = io.BytesIO()
        sf.write(buf, audio_array, sr, format="WAV")
        buf.seek(0)
        audio_b64 = base64.b64encode(buf.read()).decode("utf-8")

        return GenerateResponse(
            audio_b64=audio_b64,
            sample_rate=sr,
            platform=PLATFORM,
            duration_seconds=round(duration, 2),
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Generation failed: {str(e)}")

In [ ]:
# ── Cell 4: Start Cloudflared + Uvicorn (Kaggle) ─────────────────
import subprocess, threading, time, uvicorn, re

# Kill any existing cloudflared process first
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)  # wait for process to fully die

# Install cloudflared
subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
print("✅ cloudflared installed")

# Start tunnel and capture URL
tunnel_url = None

def run_tunnel():
    global tunnel_url
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    for line in proc.stdout:
        line = line.decode()
        match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            print("=" * 60)
            print(f"  ✅ KAGGLE API URL : {tunnel_url}")
            print(f"  🔍 Health Check   : {tunnel_url}/")
            print(f"  🎙️  Generate Voice : {tunnel_url}/generate")
            print("=" * 60)
            print("  ⚠️  Copy this URL into your frontend SERVERS config!")
            print("=" * 60)

threading.Thread(target=run_tunnel, daemon=True).start()
time.sleep(8)

# Keep-alive heartbeat
def heartbeat():
    while True:
        time.sleep(600)
        print("♥ Kaggle heartbeat")
threading.Thread(target=heartbeat, daemon=True).start()

# Start server
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

✅ cloudflared installed
  ✅ KAGGLE API URL : https://neither-fairfield-hub-jose.trycloudflare.com
  🔍 Health Check   : https://neither-fairfield-hub-jose.trycloudflare.com/
  🎙️  Generate Voice : https://neither-fairfield-hub-jose.trycloudflare.com/generate
  ⚠️  Copy this URL into your frontend SERVERS config!


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


♥ Kaggle heartbeat
♥ Kaggle heartbeat
♥ Kaggle heartbeat
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "OPTIONS / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "OPTIONS /generate HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
♥ Kaggle heartbeat
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK


...

INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
♥ Kaggle heartbeat
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
  ✅ KAGGLE API URL : https://neither-fairfield-hub-jose.trycloudflare.com
  🔍 Health Check   : https://neither-fairfield-hub-jose.trycloudflare.com/
  🎙️  Generate Voice : https://neither-fairfield-hub-jose.trycloudflare.com/generate
  ⚠️  Copy this URL into your frontend SERVERS config!
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
♥ Kaggle heartbeat
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "OPTIONS / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
  ✅ KAGGLE API URL : https://neither-fairfield-hub-jose.trycloudflare.com
  🔍 Health Check   : https://neither-fairfield-hub-jose.trycloudflare.com/
  🎙️  Generate Voice : https://neither-fairfield-hub-jose.trycloudflare.com/generate
  ⚠️  Copy this URL into your frontend SERVERS config!
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
♥ Kaggle heartbeat
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
♥ Kaggle heartbeat
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
INFO:     2409:40d0:134d:334b:51c6:ec3b:9ef1:6a88:0 - "GET / HTTP/1.1" 200 OK
♥ Kaggle heartbea